# LeetCode 1485 — Clone Binary Tree With Random Pointer bằng Python

Link đề: https://leetcode.com/problems/clone-binary-tree-with-random-pointer/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search, Binary Tree

> **Ghi chú Premium:** đây là bài Premium 🔒. Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1485/). Nếu có tài khoản Premium, bạn nên đối chiếu lại đề gốc.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
# Definition for Node.
# class Node:
#     def __init__(self, val=0, left=None, right=None, random=None):
#         self.val = val
#         self.left = left
#         self.right = right
#         self.random = random

class Solution:
    def copyRandomBinaryTree(self, root: 'Optional[Node]') -> 'Optional[NodeCopy]':
        ...
```

Lớp `NodeCopy` có **cùng thuộc tính và cùng hàm khởi tạo** với `Node`; LeetCode yêu cầu trả về cây gồm các nút `NodeCopy`.

## 1. Mục tiêu học

1. Hiểu sự khác nhau giữa **sao chép nông** (shallow copy — vẫn dùng chung nút cũ) và **sao chép sâu** (deep copy — mọi nút đều mới).
2. Biết vì sao con trỏ `random` làm bài khó hơn sao chép cây thường: nó có thể trỏ **lên trên, sang nhánh khác, hoặc vào chính nút đó** ⇒ có thể tạo **chu trình**.
3. Nắm kỹ thuật cốt lõi: **từ điển ánh xạ nút cũ → nút mới** (`dict` dùng chính đối tượng làm khoá).
4. Viết được 4 cách: baseline `O(n²)` theo vị trí, hai lượt với `dict`, DFS đệ quy có memo (cách của doocs), BFS lặp.
5. Biết cách **kiểm tra** một bản sao là sâu thật: không nút nào trùng `id()` với cây gốc, `random` của bản sao trỏ vào bản sao, cấu trúc + giá trị khớp.

## 2. Đề bài & ví dụ

Cho một cây nhị phân, mỗi nút ngoài `left`, `right` còn có thêm con trỏ `random` — có thể trỏ tới **bất kỳ nút nào trong cây** hoặc `None`. Hãy trả về **bản sao sâu** (deep copy) của cây.

**Cách biểu diễn input/output:** giống cây nhị phân thường (level-order, có `null`), nhưng mỗi nút là cặp `[val, random_index]`:
- `val`: giá trị nút;
- `random_index`: **chỉ số trong mảng input** (tính cả các vị trí `null`) của nút mà `random` trỏ tới, hoặc `null`.

Input là cây gồm các nút lớp `Node`; output phải là cây gồm các nút lớp `NodeCopy` (giống hệt `Node`).

**Ví dụ 1:** `root = [[1,null],null,[4,3],[7,0]]` → `[[1,null],null,[4,3],[7,0]]`
Cây là `[1,null,4,7]`. `random` của 4 trỏ tới nút 7 (chỉ số 3), `random` của 7 trỏ về nút 1 (chỉ số 0).

**Ví dụ 2:** `root = [[1,4],null,[1,0],null,[1,5],[1,5]]` → giống input. `random` có thể trỏ vào **chính nút đó** (nút ở chỉ số 5 trỏ về 5). Chú ý: mọi giá trị đều là `1`!

**Ví dụ 3:** `root = [[1,6],[2,5],[3,4],[4,3],[5,2],[6,1],[7,0]]` → giống input.

**Ràng buộc:** số nút trong `[0, 1000]`; `1 <= Node.val <= 10^6`.

Vì output được LeetCode tuần tự hoá lại thành mảng, đáp án đúng luôn **trùng với input** — cái khó nằm ở chỗ các nút phải là nút **mới**.

In [1]:
import random
import sys
import time
from collections import deque
from typing import Optional

class Node:
    def __init__(self, val=0, left=None, right=None, random=None):
        self.val = val
        self.left = left
        self.right = right
        self.random = random
    def __repr__(self):
        return f"Node({self.val})"

class NodeCopy:
    def __init__(self, val=0, left=None, right=None, random=None):
        self.val = val
        self.left = left
        self.right = right
        self.random = random
    def __repr__(self):
        return f"NodeCopy({self.val})"

examples = [
    [[1, None], None, [4, 3], [7, 0]],
    [[1, 4], None, [1, 0], None, [1, 5], [1, 5]],
    [[1, 6], [2, 5], [3, 4], [4, 3], [5, 2], [6, 1], [7, 0]],
]
for e in examples:
    print(e)

[[1, None], None, [4, 3], [7, 0]]
[[1, 4], None, [1, 0], None, [1, 5], [1, 5]]
[[1, 6], [2, 5], [3, 4], [4, 3], [5, 2], [6, 1], [7, 0]]


Để chạy thử ta cần hai hàm phụ (tự dựng, LeetCode không cho sẵn):

- `build_random_tree(data, cls)`: dựng cây từ mảng `[val, random_index]` theo level-order. Ta ghi lại `nodes[i]` = nút ở vị trí `i` của mảng để nối `random` ở bước sau.
- `serialize(root)`: đi BFS, ghi cả `None`, bỏ `None` thừa ở cuối, rồi đổi `random` thành chỉ số. Nếu `random` trỏ tới một nút **không thuộc cây** (dấu hiệu sao chép nông), ta ghi `'NGOÀI'` thay vì báo lỗi.

In [2]:
def build_random_tree(data, cls=Node):
    if not data or data[0] is None:
        return None
    nodes = [None] * len(data)
    root = cls(data[0][0])
    nodes[0] = root
    q = deque([root])
    i = 1
    while q and i < len(data):
        cur = q.popleft()
        if i < len(data) and data[i] is not None:
            cur.left = cls(data[i][0]); nodes[i] = cur.left; q.append(cur.left)
        i += 1
        if i < len(data) and data[i] is not None:
            cur.right = cls(data[i][0]); nodes[i] = cur.right; q.append(cur.right)
        i += 1
    for idx, item in enumerate(data):
        if item is not None and item[1] is not None:
            nodes[idx].random = nodes[item[1]]
    return root

def serialize(root):
    if root is None:
        return []
    order = []
    q = deque([root])
    while q:
        cur = q.popleft()
        order.append(cur)
        if cur is not None:
            q.append(cur.left)
            q.append(cur.right)
    while order and order[-1] is None:
        order.pop()
    pos = {id(nd): i for i, nd in enumerate(order) if nd is not None}
    out = []
    for nd in order:
        if nd is None:
            out.append(None)
        elif nd.random is None:
            out.append([nd.val, None])
        else:
            out.append([nd.val, pos.get(id(nd.random), "NGOÀI")])
    return out

for e in examples:
    t = build_random_tree(e)
    print(serialize(t) == e, serialize(t))

True [[1, None], None, [4, 3], [7, 0]]
True [[1, 4], None, [1, 0], None, [1, 5], [1, 5]]
True [[1, 6], [2, 5], [3, 4], [4, 3], [5, 2], [6, 1], [7, 0]]


## 3. Kiến thức nền

### 3.1 Biến trong Python là "nhãn" trỏ tới đối tượng

`b = a` **không** tạo đối tượng mới: `a` và `b` cùng trỏ tới một đối tượng. `id(x)` cho biết "địa chỉ" của đối tượng; `a is b` kiểm tra hai nhãn có cùng trỏ một đối tượng không.

In [3]:
a = Node(5)
b = a                 # chỉ là thêm một cái tên
c = Node(5)           # đối tượng mới, cùng giá trị
print("a is b:", a is b, "| a is c:", a is c)
print("id(a)==id(b):", id(a) == id(b), "| id(a)==id(c):", id(a) == id(c))
b.val = 99
print("sau khi sửa b.val -> a.val =", a.val)

a is b: True | a is c: False
id(a)==id(b): True | id(a)==id(c): False
sau khi sửa b.val -> a.val = 99


### 3.2 Sao chép nông vs sao chép sâu

- **Nông**: tạo vỏ mới nhưng các "con" vẫn là đối tượng cũ. Sửa bản sao có thể làm hỏng bản gốc.
- **Sâu**: mọi đối tượng bên trong cũng được tạo mới.

Module `copy` có `copy.copy` (nông) và `copy.deepcopy` (sâu). LeetCode muốn ta **tự** làm deep copy, và còn phải đổi lớp `Node` → `NodeCopy`, nên không dùng được `deepcopy`.

In [4]:
import copy
t = build_random_tree(examples[0])
shallow = copy.copy(t)
deep = copy.deepcopy(t)
print("nông: gốc mới?", shallow is not t, "| con phải dùng chung?", shallow.right is t.right)
print("sâu : gốc mới?", deep is not t, "| con phải dùng chung?", deep.right is t.right)
print("nhưng deepcopy vẫn giữ lớp:", type(deep).__name__, "(LeetCode cần NodeCopy)")

nông: gốc mới? True | con phải dùng chung? True
sâu : gốc mới? True | con phải dùng chung? False
nhưng deepcopy vẫn giữ lớp: Node (LeetCode cần NodeCopy)


### 3.3 Dùng đối tượng làm khoá `dict`

Một đối tượng của lớp tự định nghĩa (không viết `__eq__`/`__hash__`) **hash theo danh tính**: hai nút khác nhau luôn là hai khoá khác nhau, kể cả khi cùng `val`. Đây chính là thứ ta cần cho bảng ánh xạ "nút cũ → nút mới".

Mẹo: `d.get(None)` trả về `None` (vì `None` không phải khoá) — rất tiện khi nối `left/right/random` có thể rỗng.

In [5]:
n1, n2 = Node(1), Node(1)        # cùng val
mp = {n1: "bản sao của n1", n2: "bản sao của n2"}
print(len(mp), mp[n1], "|", mp[n2])
print("mp.get(None) =", mp.get(None))

2 bản sao của n1 | bản sao của n2
mp.get(None) = None


### 3.4 BFS với `deque` và DFS bằng ngăn xếp

`deque.popleft()` là `O(1)` (khác `list.pop(0)` là `O(n)`). DFS lặp dùng `list` làm stack: `append` + `pop`.

In [6]:
t = build_random_tree(examples[2])
q = deque([t]); bfs = []
while q:
    x = q.popleft(); bfs.append(x.val)
    for ch in (x.left, x.right):
        if ch: q.append(ch)
st = [t]; dfs = []
while st:
    x = st.pop(); dfs.append(x.val)
    for ch in (x.right, x.left):      # đẩy phải trước để trái ra trước
        if ch: st.append(ch)
print("BFS:", bfs, "| DFS tiền thứ tự:", dfs)

BFS: [1, 2, 3, 4, 5, 6, 7] | DFS tiền thứ tự: [1, 2, 4, 5, 3, 6, 7]


### 3.5 Giới hạn đệ quy

Python mặc định cho đệ quy sâu khoảng 1000 tầng. Cây lệch 1000 nút có thể chạm giới hạn này. LeetCode thường đã nâng giới hạn, nhưng khi tự test ta có thể dùng `sys.setrecursionlimit`, hoặc viết bản **lặp**.

In [7]:
print("giới hạn hiện tại:", sys.getrecursionlimit())
sys.setrecursionlimit(10000)
print("đã nâng lên:", sys.getrecursionlimit())

giới hạn hiện tại: 1000
đã nâng lên: 10000


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: cây `1 → (phải) 4 → (trái) 7`, `random(4) = 7`, `random(7) = 1`.

Nếu sao chép như cây thường (chỉ theo `left/right`), ta có `1', 4', 7'`. Khi tới `4'`, ta cần `4'.random` = **bản sao của 7**, tức `7'` — không phải `7` gốc! Vấn đề: lúc đang ở `4`, có thể `7'` **chưa được tạo**; và với `7`, `random` trỏ ngược về `1` mà `1'` **đã tạo rồi** — nếu tạo thêm một `1'` nữa thì cây bị nhân đôi nút.

⇒ Cần một "sổ ghi chép" `seen`: `seen[nút gốc] = bản sao`. Quy tắc:
- Gặp nút chưa có trong sổ ⇒ tạo bản sao, ghi vào sổ.
- Gặp nút đã có ⇒ dùng lại bản sao trong sổ.

| bước | nút gốc | sổ `seen` sau bước | ghi chú |
|---|---|---|---|
| 1 | 1 | {1→1'} | `1.random = None` |
| 2 | 4 | {1→1', 4→4'} | `1'.right = 4'` |
| 3 | 7 | {…, 7→7'} | `4'.left = 7'` |
| 4 | — | — | `4'.random = seen[7] = 7'`, `7'.random = seen[1] = 1'` |

In [8]:
t = build_random_tree(examples[0])
seen = {}
for old in [t, t.right, t.right.left]:          # 1, 4, 7
    seen[old] = NodeCopy(old.val)
    print("tạo bản sao cho", old, "-> sổ có", len(seen), "mục")
for old, new in seen.items():
    new.left, new.right, new.random = seen.get(old.left), seen.get(old.right), seen.get(old.random)
print(serialize(seen[t]))

tạo bản sao cho Node(1) -> sổ có 1 mục
tạo bản sao cho Node(4) -> sổ có 2 mục
tạo bản sao cho Node(7) -> sổ có 3 mục
[[1, None], None, [4, 3], [7, 0]]


## 5. Cách 1 — Brute Force / baseline: sao chép khung rồi tìm `random` theo vị trí

**Ý tưởng:** không dùng `dict`.
1. Sao chép **khung cây** (chỉ `val`, `left`, `right`) bằng đệ quy như cây thường.
2. Liệt kê nút của hai cây theo **cùng một thứ tự** (tiền thứ tự). Vì hai cây cùng hình dạng, `olds[i]` ↔ `news[i]`.
3. Với mỗi `olds[i]` có `random`, **dò tuyến tính** trong `olds` để tìm chỉ số `j` của nút đích (so bằng `is`), rồi đặt `news[i].random = news[j]`.

In [9]:
def preorder(root):
    out, st = [], [root] if root else []
    while st:
        x = st.pop()
        out.append(x)
        if x.right: st.append(x.right)
        if x.left: st.append(x.left)
    return out

def clone_brute(root):
    def copy_struct(node):
        if node is None:
            return None
        c = NodeCopy(node.val)
        c.left = copy_struct(node.left)
        c.right = copy_struct(node.right)
        return c
    new_root = copy_struct(root)
    olds, news = preorder(root), preorder(new_root)
    for i, o in enumerate(olds):
        if o.random is not None:
            j = next(j for j, x in enumerate(olds) if x is o.random)   # dò O(n)
            news[i].random = news[j]
    return new_root

for e in examples:
    print(serialize(clone_brute(build_random_tree(e))) == e)

True
True
True


**Độ phức tạp:** sao chép khung `O(n)`, nhưng mỗi `random` dò `O(n)` ⇒ **`O(n²)`** thời gian, `O(n)` bộ nhớ (hai danh sách + đệ quy).

**Nút thắt:** câu hỏi "nút `X` nằm ở vị trí nào / bản sao của `X` là gì?" bị trả lời bằng tìm kiếm tuyến tính. Với `n = 1000` vẫn qua (10⁶ phép so), nhưng ý tưởng không mở rộng được. Nếu có **bảng tra `O(1)`** "nút cũ → nút mới" thì xong.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay vì dò vị trí, lưu trực tiếp `seen[old] = new` trong một `dict`. Tra cứu `O(1)` trung bình.

**Invariant (bất biến):** tại mọi thời điểm, mỗi nút gốc có **nhiều nhất một** bản sao, và bản sao đó nằm trong `seen`.

**Quy tắc "nếu… thì…":**
- Nếu `old is None` thì bản sao là `None`.
- Nếu `old in seen` thì trả `seen[old]` (không tạo thêm — chặn chu trình và chặn nhân đôi).
- Ngược lại: tạo `new = NodeCopy(old.val)`, **ghi vào `seen` TRƯỚC** khi xử lý con/random, rồi nối `new.left/right/random` bằng bản sao của `old.left/right/random`.

Ghi vào sổ trước khi đệ quy là chi tiết then chốt: nếu `random` vòng ngược lại chính nút đang xử lý (ví dụ 2: nút trỏ về chính nó), lần gọi sau sẽ thấy nút đã có trong sổ và dừng.

## 7. Các cách tiếp theo

### Cách 2 — Hai lượt với `dict`

**Lượt 1:** duyệt cây (DFS bằng stack) và tạo bản sao "trơ trọi" (chưa nối gì) cho **mọi** nút: `mp[old] = NodeCopy(old.val)`.
**Lượt 2:** với mỗi cặp `(old, new)` trong `mp`, nối `new.left = mp.get(old.left)`, tương tự `right`, `random`. Nhờ `mp.get(None) = None` mà không cần `if`.

Ưu điểm: dễ hiểu nhất, không đệ quy, không lo thứ tự tạo nút.

In [10]:
def clone_two_pass(root):
    if root is None:
        return None
    mp = {}
    st = [root]
    while st:                                   # lượt 1: tạo nút
        o = st.pop()
        mp[o] = NodeCopy(o.val)
        if o.left: st.append(o.left)
        if o.right: st.append(o.right)
    for o, c in mp.items():                      # lượt 2: nối con trỏ
        c.left = mp.get(o.left)
        c.right = mp.get(o.right)
        c.random = mp.get(o.random)
    return mp[root]

for e in examples:
    print(serialize(clone_two_pass(build_random_tree(e))) == e)

True
True
True


Code nộp LeetCode (Cách 2):

In [11]:
class SolutionTwoPass:
    def copyRandomBinaryTree(self, root: 'Optional[Node]') -> 'Optional[NodeCopy]':
        if root is None:
            return None
        mp = {}
        st = [root]
        while st:
            o = st.pop()
            mp[o] = NodeCopy(o.val)
            if o.left:
                st.append(o.left)
            if o.right:
                st.append(o.right)
        for o, c in mp.items():
            c.left = mp.get(o.left)
            c.right = mp.get(o.right)
            c.random = mp.get(o.random)
        return mp[root]

print(serialize(SolutionTwoPass().copyRandomBinaryTree(build_random_tree(examples[1]))))

[[1, 4], None, [1, 0], None, [1, 5], [1, 5]]


**Độ phức tạp:** `O(n)` thời gian (mỗi nút vào stack một lần, mỗi cặp nối một lần), `O(n)` bộ nhớ cho `mp` và stack.

### Cách 3 — DFS đệ quy có memo (cách của doocs)

`dfs(old)` trả về bản sao của `old`, áp dụng đúng 3 quy tắc ở mục 6. Đệ quy theo **ba** hướng: `left`, `right`, `random`. Hướng `random` có thể "nhảy" sang nút chưa thăm — không sao: nút đó được tạo luôn, và sau này khi duyệt tới bằng `left/right` thì sổ đã có, chỉ việc dùng lại.

In [12]:
def clone_dfs(root):
    seen = {}
    def dfs(old):
        if old is None:
            return None
        if old in seen:
            return seen[old]
        new = NodeCopy(old.val)
        seen[old] = new                 # ghi sổ TRƯỚC khi đệ quy
        new.left = dfs(old.left)
        new.right = dfs(old.right)
        new.random = dfs(old.random)
        return new
    return dfs(root)

for e in examples:
    print(serialize(clone_dfs(build_random_tree(e))) == e)

True
True
True


Code nộp LeetCode (Cách 3):

In [13]:
class SolutionDFS:
    def copyRandomBinaryTree(self, root: 'Optional[Node]') -> 'Optional[NodeCopy]':
        def dfs(root: Optional[Node]) -> Optional[NodeCopy]:
            if root is None:
                return None
            if root in seen:
                return seen[root]
            copy = NodeCopy(root.val)
            seen[root] = copy
            copy.left = dfs(root.left)
            copy.right = dfs(root.right)
            copy.random = dfs(root.random)
            return copy

        seen = {}
        return dfs(root)

print(serialize(SolutionDFS().copyRandomBinaryTree(build_random_tree(examples[2]))))

[[1, 6], [2, 5], [3, 4], [4, 3], [5, 2], [6, 1], [7, 0]]


**Độ phức tạp:** mỗi nút được tạo đúng một lần, mỗi lời gọi `dfs` ngoài lần tạo chỉ là một lần tra sổ `O(1)`; có `≤ 3n + 1` lời gọi ⇒ `O(n)` thời gian. Bộ nhớ `O(n)` cho `seen` + độ sâu đệ quy `O(n)` (trường hợp xấu: cây lệch hoặc chuỗi `random` dài).

### Cách 4 — BFS lặp (không đệ quy)

Duyệt BFS **chỉ theo cạnh cây** (`left/right`) để mỗi nút được lấy ra khỏi hàng đợi đúng một lần. Hàm `get(old)` trả về bản sao, tạo mới nếu chưa có. Khi lấy `old` ra: nối `new.left = get(old.left)`, `new.right = get(old.right)`, `new.random = get(old.random)`.

In [14]:
def clone_bfs(root):
    if root is None:
        return None
    mp = {}
    def get(old):
        if old is None:
            return None
        if old not in mp:
            mp[old] = NodeCopy(old.val)
        return mp[old]
    q = deque([root])
    get(root)
    while q:
        old = q.popleft()
        new = mp[old]
        new.left = get(old.left)
        new.right = get(old.right)
        new.random = get(old.random)
        if old.left: q.append(old.left)
        if old.right: q.append(old.right)
    return mp[root]

for e in examples:
    print(serialize(clone_bfs(build_random_tree(e))) == e)

True
True
True


Code nộp LeetCode (Cách 4):

In [15]:
class SolutionBFS:
    def copyRandomBinaryTree(self, root: 'Optional[Node]') -> 'Optional[NodeCopy]':
        if root is None:
            return None
        mp = {}

        def get(old):
            if old is None:
                return None
            if old not in mp:
                mp[old] = NodeCopy(old.val)
            return mp[old]

        q = deque([root])
        get(root)
        while q:
            old = q.popleft()
            new = mp[old]
            new.left = get(old.left)
            new.right = get(old.right)
            new.random = get(old.random)
            if old.left:
                q.append(old.left)
            if old.right:
                q.append(old.right)
        return mp[root]

print(serialize(SolutionBFS().copyRandomBinaryTree(build_random_tree(examples[0]))))

[[1, None], None, [4, 3], [7, 0]]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (`mp` + hàng đợi tối đa bằng độ rộng cây). Không lo giới hạn đệ quy.

## 8. TRACE từng bước

Trace Cách 3 (DFS memo) trên **Ví dụ 2** `[[1,4],null,[1,0],null,[1,5],[1,5]]` — mọi giá trị đều là 1 nên ta đặt tên nút theo **chỉ số mảng**: `#0, #2, #4, #5`. Cây: `#0 → phải #2 → phải #4`, `#4` có trái `#5`... (xem `serialize`). Random: `#0→#4`, `#2→#0`, `#4→#5`, `#5→#5`.

Bảng dự kiến (rút gọn):

| lời gọi | hướng | kết quả | sổ `seen` |
|---|---|---|---|
| dfs(#0) | gốc | tạo #0' | {#0} |
| dfs(None) | #0.left | None | |
| dfs(#2) | #0.right | tạo #2' | {#0,#2} |
| dfs(#4) | #2.right | tạo #4' | {#0,#2,#4} |
| dfs(#5) | #4.left | tạo #5' | {#0,#2,#4,#5} |
| dfs(#5) | #5.random | **đã có** (trỏ chính nó) | |
| dfs(#0) | #2.random | **đã có** → #0' | |

In [16]:
def trace_dfs(data):
    root = build_random_tree(data)
    names = {}
    # đặt tên theo chỉ số mảng
    order = []
    q = deque([root])
    while q:
        x = q.popleft(); order.append(x)
        if x is not None:
            q.append(x.left); q.append(x.right)
    for i, x in enumerate(order):
        if x is not None:
            names[x] = f"#{i}"
    seen = {}
    def dfs(old, via, depth):
        pad = "  " * depth
        if old is None:
            print(f"{pad}dfs(None) qua {via} -> None")
            return None
        if old in seen:
            print(f"{pad}dfs({names[old]}) qua {via} -> ĐÃ CÓ, dùng lại {names[old]}'")
            return seen[old]
        new = NodeCopy(old.val)
        seen[old] = new
        print(f"{pad}dfs({names[old]}) qua {via} -> TẠO {names[old]}' | sổ = {[names[k] for k in seen]}")
        new.left = dfs(old.left, names[old] + ".left", depth + 1)
        new.right = dfs(old.right, names[old] + ".right", depth + 1)
        new.random = dfs(old.random, names[old] + ".random", depth + 1)
        return new
    res = dfs(root, "gốc", 0)
    print("kết quả:", serialize(res))
    return res

_ = trace_dfs(examples[1])

dfs(#0) qua gốc -> TẠO #0' | sổ = ['#0']
  dfs(None) qua #0.left -> None
  dfs(#2) qua #0.right -> TẠO #2' | sổ = ['#0', '#2']
    dfs(None) qua #2.left -> None
    dfs(#4) qua #2.right -> TẠO #4' | sổ = ['#0', '#2', '#4']
      dfs(#5) qua #4.left -> TẠO #5' | sổ = ['#0', '#2', '#4', '#5']
        dfs(None) qua #5.left -> None
        dfs(None) qua #5.right -> None
        dfs(#5) qua #5.random -> ĐÃ CÓ, dùng lại #5'
      dfs(None) qua #4.right -> None
      dfs(#5) qua #4.random -> ĐÃ CÓ, dùng lại #5'
    dfs(#0) qua #2.random -> ĐÃ CÓ, dùng lại #0'
  dfs(#4) qua #0.random -> ĐÃ CÓ, dùng lại #4'
kết quả: [[1, 4], None, [1, 0], None, [1, 5], [1, 5]]


Quan sát: mỗi nút chỉ có một dòng "TẠO"; các lần sau đều "ĐÃ CÓ". Ở `#5.random` (tự trỏ), vì `#5` đã ghi sổ trước khi đệ quy nên không lặp vô hạn.

Trace thêm Cách 2 (hai lượt) trên Ví dụ 1 để thấy rõ trạng thái `mp` và các phép nối:

In [17]:
def trace_two_pass(data):
    root = build_random_tree(data)
    mp = {}
    st = [root]
    step = 0
    while st:
        o = st.pop(); step += 1
        mp[o] = NodeCopy(o.val)
        print(f"Lượt 1 - bước {step}: lấy {o} ra, tạo bản sao; stack còn {st}, mp có {len(mp)} mục")
        if o.left: st.append(o.left)
        if o.right: st.append(o.right)
    for o, c in mp.items():
        c.left, c.right, c.random = mp.get(o.left), mp.get(o.right), mp.get(o.random)
        print(f"Lượt 2: {o}: left={c.left}, right={c.right}, random={c.random}")
    print("kết quả:", serialize(mp[root]))

trace_two_pass(examples[0])

Lượt 1 - bước 1: lấy Node(1) ra, tạo bản sao; stack còn [], mp có 1 mục
Lượt 1 - bước 2: lấy Node(4) ra, tạo bản sao; stack còn [], mp có 2 mục
Lượt 1 - bước 3: lấy Node(7) ra, tạo bản sao; stack còn [], mp có 3 mục
Lượt 2: Node(1): left=None, right=NodeCopy(4), random=None
Lượt 2: Node(4): left=NodeCopy(7), right=None, random=NodeCopy(7)
Lượt 2: Node(7): left=None, right=None, random=NodeCopy(1)
kết quả: [[1, None], None, [4, 3], [7, 0]]


## 9. Code nộp LeetCode

Bản chính dùng DFS có memo (ngắn, chuẩn, giống doocs). Trên LeetCode **không** cần định nghĩa `Node`/`NodeCopy` (đã có sẵn), và `Optional` đã được import sẵn.

In [18]:
class Solution:
    def copyRandomBinaryTree(self, root: 'Optional[Node]') -> 'Optional[NodeCopy]':
        seen = {}

        def dfs(node):
            if node is None:
                return None
            if node in seen:
                return seen[node]
            copy = NodeCopy(node.val)
            seen[node] = copy
            copy.left = dfs(node.left)
            copy.right = dfs(node.right)
            copy.random = dfs(node.random)
            return copy

        return dfs(root)

for e in examples:
    print(serialize(Solution().copyRandomBinaryTree(build_random_tree(e))))

[[1, None], None, [4, 3], [7, 0]]
[[1, 4], None, [1, 0], None, [1, 5], [1, 5]]
[[1, 6], [2, 5], [3, 4], [4, 3], [5, 2], [6, 1], [7, 0]]


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (mỗi nút gốc có đúng một bản sao).** Một bản sao chỉ được tạo khi `old not in seen`, và ngay lập tức được ghi vào `seen`. Mọi lần gặp lại `old` đều trả `seen[old]`. Vì vậy không bao giờ có hai bản sao cho cùng một nút gốc.

**Bổ đề 2 (mọi nút gốc đều được sao).** Gốc được gọi `dfs`. Mỗi nút đã tạo bản sao đều gọi `dfs` cho `left` và `right` ⇒ theo quy nạp trên độ sâu, mọi nút trong cây (đều tới được từ gốc qua `left/right`) đều được gọi ⇒ có bản sao.

**Bổ đề 3 (con trỏ đúng).** Với mỗi nút gốc `x` có bản sao `x'`: `x'.left = dfs(x.left) = (x.left)'`, tương tự `right`, `random`. Vậy ánh xạ `x ↦ x'` là một **đẳng cấu**: giữ nguyên `val` và mọi con trỏ. Hai cây đẳng cấu thì tuần tự hoá giống nhau.

**Dừng.** Số lần "TẠO" ≤ n; mỗi lần TẠO sinh ra đúng 3 lời gọi con; lời gọi không tạo thì trả ngay. Tổng số lời gọi ≤ 3n + 1 ⇒ luôn dừng, kể cả khi `random` tạo chu trình.

**Sâu thật:** mọi nút trong kết quả đều do `NodeCopy(...)` tạo ra, và mọi con trỏ của chúng chỉ trỏ tới giá trị lấy từ `seen` (toàn nút mới) hoặc `None` ⇒ không có tham chiếu nào về cây gốc.

## 11. Phân tích Time / Space

Gọi `n` = số nút (`0 ≤ n ≤ 1000`), `h` = chiều cao cây, `w` = độ rộng lớn nhất.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Baseline theo vị trí | `O(n²)` (dò tuyến tính cho mỗi `random`) | `O(n)` |
| 2. Hai lượt + dict | `O(n)` | `O(n)` dict + `O(n)` stack |
| 3. DFS memo | `O(n)` | `O(n)` dict + `O(độ sâu đệ quy)` ≤ `O(n)` |
| 4. BFS | `O(n)` | `O(n)` dict + `O(w)` hàng đợi |

Thao tác `dict` là `O(1)` **trung bình** (băm theo danh tính đối tượng). Không tính bộ nhớ của chính cây kết quả (bắt buộc `n` nút mới).

Lưu ý với DFS: độ sâu đệ quy không chỉ là `h` — nhánh `random` có thể dẫn sâu thêm, trường hợp xấu nhất ~`n` tầng.

## 12. Bộ test

### 12.1 Hàm kiểm tra bản sao sâu

`check_deep_copy(orig, cp)` kiểm tra:
1. Không nút nào của bản sao trùng `id()` với một nút của cây gốc.
2. Mọi nút của bản sao là `NodeCopy`.
3. Mọi `random` của bản sao trỏ vào một nút **thuộc bản sao**.
4. `serialize` hai cây giống nhau (cấu trúc, giá trị, chỉ số `random`).
5. Số nút bằng nhau (không có nút "thừa" lơ lửng chỉ được `random` trỏ tới).

In [19]:
def collect(root):
    # có tập "đã thăm" để an toàn cả khi bản sao (sai) có vòng lặp
    out, st, vis = [], [root] if root else [], set()
    while st:
        x = st.pop()
        if id(x) in vis:
            continue
        vis.add(id(x)); out.append(x)
        if x.left: st.append(x.left)
        if x.right: st.append(x.right)
    return out

def check_deep_copy(orig, cp):
    o_nodes, c_nodes = collect(orig), collect(cp)
    o_ids = {id(x) for x in o_nodes}
    c_ids = {id(x) for x in c_nodes}
    if len(o_nodes) != len(c_nodes):
        return False, "số nút khác nhau"
    if o_ids & c_ids:
        return False, "có nút dùng chung với cây gốc"
    if any(type(x) is not NodeCopy for x in c_nodes):
        return False, "có nút không phải NodeCopy"
    if any(x.random is not None and id(x.random) not in c_ids for x in c_nodes):
        return False, "random trỏ ra ngoài bản sao"
    if serialize(orig) != serialize(cp):
        return False, "cấu trúc/giá trị/random không khớp"
    return True, "OK"

t = build_random_tree(examples[0])
print(check_deep_copy(t, Solution().copyRandomBinaryTree(t)))
print(check_deep_copy(t, t), "(chính nó -> phải bị bắt)")

(True, 'OK')
(False, 'có nút dùng chung với cây gốc') (chính nó -> phải bị bắt)


### 12.2 Ví dụ đề + edge case

In [20]:
ALL = {
    "brute": clone_brute,
    "two_pass": clone_two_pass,
    "dfs": clone_dfs,
    "bfs": clone_bfs,
    "sub_two_pass": lambda r: SolutionTwoPass().copyRandomBinaryTree(r),
    "sub_dfs": lambda r: SolutionDFS().copyRandomBinaryTree(r),
    "sub_bfs": lambda r: SolutionBFS().copyRandomBinaryTree(r),
    "submit": lambda r: Solution().copyRandomBinaryTree(r),
}
edge = [
    [],                                   # cây rỗng
    [[5, None]],                          # một nút
    [[5, 0]],                             # một nút tự trỏ
    [[1, 1], [2, 0]],                     # hai nút trỏ chéo nhau (chu trình)
    [[1, 2], None, [2, 4], None, [3, 0]], # cây lệch phải, random vòng về gốc
    [[7, None], [7, None], [7, None]],    # trùng giá trị, không random
]
for name, f in ALL.items():
    for data in examples + edge:
        orig = build_random_tree(data)
        before = serialize(orig)
        cp = f(orig)
        if orig is None:
            assert cp is None, name
            continue
        ok, why = check_deep_copy(orig, cp)
        assert ok, (name, data, why)
        assert serialize(orig) == before, (name, "cây gốc bị sửa!")
print("Mọi cách đúng với", len(examples + edge), "test; cây gốc không bị thay đổi")

Mọi cách đúng với 9 test; cây gốc không bị thay đổi


### 12.3 Cây ngẫu nhiên (seed cố định)

Sinh cây có `n` nút: mỗi nút mới gắn vào một **chỗ trống ngẫu nhiên** (con trái/phải còn `None`) của các nút đã có ⇒ đủ loại hình dạng. `random` của mỗi nút chọn ngẫu nhiên một nút bất kỳ (kể cả chính nó) hoặc `None`. Giá trị lấy trong khoảng nhỏ để có nhiều giá trị trùng.

In [21]:
def random_tree(rng, n, maxv=5):
    if n == 0:
        return None
    nodes = [Node(rng.randint(1, maxv))]
    slots = [(nodes[0], "left"), (nodes[0], "right")]
    for _ in range(n - 1):
        i = rng.randrange(len(slots))
        par, side = slots[i]
        slots[i] = slots[-1]; slots.pop()
        nd = Node(rng.randint(1, maxv))
        setattr(par, side, nd)
        nodes.append(nd)
        slots += [(nd, "left"), (nd, "right")]
    for nd in nodes:
        nd.random = rng.choice(nodes + [None])
    return nodes[0]

rng = random.Random(1485)
for _ in range(400):
    orig = random_tree(rng, rng.randint(0, 40))
    ref = serialize(orig)
    for name, f in ALL.items():
        cp = f(orig)
        if orig is None:
            assert cp is None
            continue
        ok, why = check_deep_copy(orig, cp)
        assert ok, (name, why)
        assert serialize(orig) == ref
print("400 cây ngẫu nhiên: mọi cách tạo bản sao sâu hợp lệ")

400 cây ngẫu nhiên: mọi cách tạo bản sao sâu hợp lệ


### 12.4 Giới hạn `n = 1000` (cây ngẫu nhiên và cây lệch hoàn toàn)

In [22]:
def skewed(n):
    root = cur = Node(1)
    nodes = [root]
    for v in range(2, n + 1):
        cur.right = Node(v); cur = cur.right; nodes.append(cur)
    for i, nd in enumerate(nodes):
        nd.random = nodes[(i * 7) % n]
    return root

for label, orig in [("ngẫu nhiên", random_tree(rng, 1000, 10**6)), ("lệch phải", skewed(1000))]:
    for name, f in ALL.items():
        t0 = time.perf_counter()
        cp = f(orig)
        dt = time.perf_counter() - t0
        assert check_deep_copy(orig, cp)[0], (label, name)
        print(f"{label:10} | {name:13}: {dt*1000:7.2f} ms")

ngẫu nhiên | brute        :   15.72 ms
ngẫu nhiên | two_pass     :    0.36 ms
ngẫu nhiên | dfs          :    0.60 ms
ngẫu nhiên | bfs          :    0.48 ms
ngẫu nhiên | sub_two_pass :    0.36 ms
ngẫu nhiên | sub_dfs      :    0.42 ms
ngẫu nhiên | sub_bfs      :    0.75 ms
ngẫu nhiên | submit       :    0.65 ms
lệch phải  | brute        :   23.99 ms
lệch phải  | two_pass     :    0.35 ms
lệch phải  | dfs          :    0.56 ms
lệch phải  | bfs          :    0.45 ms
lệch phải  | sub_two_pass :    0.36 ms
lệch phải  | sub_dfs      :    0.80 ms
lệch phải  | sub_bfs      :    0.77 ms
lệch phải  | submit       :    0.73 ms


Baseline `O(n²)` chậm hơn rõ rệt nhưng vẫn qua với `n = 1000`. Các cách `O(n)` chỉ tốn vài mili-giây.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gán thẳng `random` của nút gốc (sao chép nông)

Viết `copy.random = node.random`: con trỏ `random` của bản sao trỏ **về cây gốc**. Output tuần tự hoá trông có vẻ đúng nếu chỉ nhìn `val`, nhưng đây không phải deep copy.

In [23]:
def clone_wrong1(root):
    if root is None:
        return None
    c = NodeCopy(root.val)
    c.left = clone_wrong1(root.left)
    c.right = clone_wrong1(root.right)
    c.random = root.random              # SAI: trỏ vào nút gốc
    return c

orig = build_random_tree(examples[0])
bad = clone_wrong1(orig)
print("serialize bản sai :", serialize(bad))
print("kiểm tra          :", check_deep_copy(orig, bad))
print("bản đúng          :", check_deep_copy(orig, Solution().copyRandomBinaryTree(orig)))

serialize bản sai : [[1, None], None, [4, 'NGOÀI'], [7, 'NGOÀI']]
kiểm tra          : (False, 'random trỏ ra ngoài bản sao')
bản đúng          : (True, 'OK')


`serialize` ghi `'NGOÀI'` vì `random` trỏ tới nút không thuộc bản sao. **Sửa:** `random` phải là **bản sao** của `node.random`, tra qua `seen`.

### Lỗi 2 — Đệ quy theo `random` nhưng không có memo

Gọi `dfs(node.random)` mà không ghi sổ: (a) nút được trỏ tới bị **tạo thêm bản thứ hai** (bản sao có nút lơ lửng ngoài cây); (b) nếu `random` tạo chu trình (nút tự trỏ) ⇒ đệ quy vô hạn. Ta bắt `RecursionError` để notebook không dừng.

In [24]:
def clone_wrong2(root):
    if root is None:
        return None
    c = NodeCopy(root.val)
    c.left = clone_wrong2(root.left)
    c.right = clone_wrong2(root.right)
    c.random = clone_wrong2(root.random)   # SAI: không memo
    return c

old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(300)
for data in [[[1, None], [2, None], [3, 1]], [[5, 0]]]:
    orig = build_random_tree(data)
    try:
        bad = clone_wrong2(orig)
        print(data, "->", serialize(bad), "|", check_deep_copy(orig, bad))
    except RecursionError:
        print(data, "-> RecursionError (đệ quy vô hạn vì chu trình random)")
sys.setrecursionlimit(old_limit)
print("bản đúng:", serialize(Solution().copyRandomBinaryTree(build_random_tree([[5, 0]]))))

[[1, None], [2, None], [3, 1]] -> [[1, None], [2, None], [3, 'NGOÀI']] | (False, 'random trỏ ra ngoài bản sao')
[[5, 0]] -> RecursionError (đệ quy vô hạn vì chu trình random)
bản đúng: [[5, 0]]


Với `[[1,null],[2,null],[3,1]]`: nút 3 có `random → 2`; bản sai tạo một nút `2` **mới** ngoài cây ⇒ `'NGOÀI'`. Với `[[5,0]]` (tự trỏ) ⇒ vô hạn. **Sửa:** dùng `seen` và ghi sổ **trước** khi đệ quy.

### Lỗi 3 — Dùng `val` làm khoá ánh xạ

Giá trị có thể **trùng** (ví dụ 2 toàn số 1). Khoá `seen[node.val]` gộp các nút khác nhau thành một ⇒ bản sao sai cấu trúc.

In [25]:
def clone_wrong3(root):
    seen = {}
    def dfs(old):
        if old is None:
            return None
        if old.val in seen:              # SAI: khoá theo giá trị
            return seen[old.val]
        new = NodeCopy(old.val)
        seen[old.val] = new
        new.left = dfs(old.left)
        new.right = dfs(old.right)
        new.random = dfs(old.random)
        return new
    return dfs(root)

orig = build_random_tree(examples[1])
bad = clone_wrong3(orig)
print("mong đợi:", serialize(orig))
print("bản sai : số nút =", len(collect(bad)), "| gốc tự làm con phải của mình?", bad.right is bad)
print("kiểm tra:", check_deep_copy(orig, bad))
print("đúng    :", check_deep_copy(orig, Solution().copyRandomBinaryTree(orig)))

mong đợi: [[1, 4], None, [1, 0], None, [1, 5], [1, 5]]
bản sai : số nút = 1 | gốc tự làm con phải của mình? True
kiểm tra: (False, 'số nút khác nhau')
đúng    : (True, 'OK')


**Sửa:** dùng **chính đối tượng nút** làm khoá (hash theo danh tính), không dùng `val`.

### Lỗi 4 — Tạo `Node` thay vì `NodeCopy`

Thuật toán đúng nhưng tạo nút bằng `Node(...)`. Trên LeetCode, checker so lớp ⇒ sai. Ta kiểm tra bằng `check_deep_copy`:

In [26]:
def clone_wrong4(root):
    seen = {}
    def dfs(old):
        if old is None:
            return None
        if old in seen:
            return seen[old]
        new = Node(old.val)               # SAI: phải là NodeCopy
        seen[old] = new
        new.left, new.right, new.random = dfs(old.left), dfs(old.right), dfs(old.random)
        return new
    return dfs(root)

orig = build_random_tree(examples[2])
print("bản sai:", check_deep_copy(orig, clone_wrong4(orig)))
print("đúng   :", check_deep_copy(orig, Solution().copyRandomBinaryTree(orig)))

bản sai: (False, 'có nút không phải NodeCopy')
đúng   : (True, 'OK')


### Lỗi 5 — Ghi sổ **sau** khi đệ quy

Viết `new.left = dfs(...)` … rồi mới `seen[old] = new`. Khi `random` vòng về nút đang xử lý, sổ chưa có ⇒ tạo bản sao thứ hai (hoặc đệ quy vô hạn với nút tự trỏ). Luôn ghi sổ **ngay sau khi tạo**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**deep copy** một cấu trúc có con trỏ phụ (`random`, `neighbors`, …)" ⇒ nghĩ ngay **`dict` cũ → mới**.
- Hai mẫu viết hay nhất: **hai lượt** (tạo hết rồi nối) — khó sai nhất; **DFS memo** — ngắn nhất.
- `mp.get(None)` trả `None` ⇒ lượt nối không cần `if`.
- Luôn **ghi sổ trước khi đệ quy** — đó là thứ chặn chu trình.
- Nếu sợ giới hạn đệ quy: dùng hai lượt hoặc BFS.
- Test deep copy: so `id()` / dùng `is`, đừng chỉ so giá trị.
- Bài cùng họ: **138 Copy List with Random Pointer**, **133 Clone Graph**, **1490 Clone N-ary Tree**, **1650**/**1660** (cây có con trỏ phụ).

## 15. Biến thể của bài

1. **LC 138 — Copy List with Random Pointer:** danh sách liên kết thay vì cây; cùng kỹ thuật `dict`, hoặc kỹ thuật "chèn nút sao xen kẽ" để đạt `O(1)` bộ nhớ phụ.
2. **LC 133 — Clone Graph:** đồ thị vô hướng có chu trình ⇒ memo là **bắt buộc**.
3. **LC 1490 — Clone N-ary Tree:** không có `random` ⇒ không cần memo, đệ quy đơn giản.
4. Đề yêu cầu **kiểm tra** hai cây có phải bản sao sâu của nhau không ⇒ chính là `check_deep_copy`: đi song song hai cây, ánh xạ nút ↔ nút và kiểm tra `random` khớp.
5. Thay `random` bằng `parent` (con trỏ cha) ⇒ cha của bản sao = bản sao của cha; vẫn dùng `dict` hoặc truyền tham số khi đệ quy.

In [27]:
# Minh hoạ biến thể 5: cây có con trỏ parent, sao chép bằng cách truyền cha mới xuống
class PNode:
    def __init__(self, val, left=None, right=None, parent=None):
        self.val, self.left, self.right, self.parent = val, left, right, parent

def clone_with_parent(node, new_parent=None):
    if node is None:
        return None
    c = PNode(node.val, parent=new_parent)
    c.left = clone_with_parent(node.left, c)
    c.right = clone_with_parent(node.right, c)
    return c

r = PNode(1); r.left = PNode(2, parent=r); r.right = PNode(3, parent=r)
cr = clone_with_parent(r)
print(cr.left.parent is cr, cr.right.parent is cr, cr.left.parent is r)

True True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Baseline vị trí | Sao khung, dò chỉ số `random` tuyến tính | `O(n²)` | `O(n)` | Khi chưa nghĩ ra `dict`; để kiểm chứng |
| 2. Hai lượt + dict | Tạo mọi nút trước, nối con trỏ sau | `O(n)` | `O(n)` | Dễ viết, khó sai, không đệ quy |
| 3. DFS memo | `dfs(x)` trả bản sao, ghi sổ trước khi đệ quy | `O(n)` | `O(n)` | Code ngắn nhất, chuẩn phỏng vấn |
| 4. BFS | Duyệt theo cạnh cây, `get()` tạo khi cần | `O(n)` | `O(n)` | Cây rất sâu, tránh đệ quy |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Với `data = [[3, 2], [4, None], [5, 0]]`, nút có giá trị bao nhiêu là `random` của gốc? Và `serialize(Solution().copyRandomBinaryTree(build_random_tree(data)))` in ra gì?

### Bài 2 (tính tay)
Với Ví dụ 3, khi chạy `clone_dfs`, liệt kê **thứ tự các nút được TẠO** (theo giá trị).

### Bài 3 (điền TODO)
Hoàn thiện `count_self_random(root)`: đếm số nút có `random` trỏ vào **chính nó**.

### Bài 4 (tự viết lại)
Viết `copy_random_list(head)` cho **LC 138** (danh sách liên kết có `random`) bằng `dict` hai lượt.

### Bài 5 (mini challenge)
Viết `is_deep_copy_pair(a, b)` **không dùng `serialize`**: đi song song hai cây, xây ánh xạ `a_node → b_node`, kiểm tra cùng hình dạng, cùng `val`, `random` khớp theo ánh xạ, và không nút nào dùng chung.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_self_random(root):
    # TODO: duyệt mọi nút, đếm nút x có x.random is x
    pass

# Bài 4 — TODO
class LNode:
    def __init__(self, val, next=None, random=None):
        self.val, self.next, self.random = val, next, random

def copy_random_list(head):
    # TODO: lượt 1 tạo bản sao cho mọi nút; lượt 2 nối next và random
    pass

# Bài 5 — TODO
def is_deep_copy_pair(a, b):
    # TODO: BFS song song, ánh xạ mp[a_node] = b_node, kiểm tra từng điều kiện
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ số 2 là nút giá trị 5. (2) Nút 5 có `random → 0` (gốc). (3) Bản sao tuần tự hoá giống hệt input.

**Bài 2.** (1) DFS đi `left` trước. (2) Gốc 1 tạo, sang trái 2, trái 4 → `4.random = 4`… chú ý `random` có thể tạo trước nút chưa tới qua cây. (3) Nút 5 có `random → 3`, nên nút 3 được tạo **trước** khi DFS quay về nhánh phải của gốc; ghi log bằng code để kiểm chứng.

**Bài 3.** (1) Dùng `collect(root)`. (2) Điều kiện `x.random is x`. (3) `sum(1 for x in ... if ...)`.

**Bài 4.** (1) `mp = {}` và duyệt `cur = head` theo `next`. (2) Lượt 2: `mp[cur].next = mp.get(cur.next)`, `mp[cur].random = mp.get(cur.random)`. (3) Trả `mp.get(head)`.

**Bài 5.** (1) Hàng đợi chứa cặp `(x, y)`. (2) Nếu một bên `None` mà bên kia không ⇒ False; `x is y` ⇒ False; `x.val != y.val` ⇒ False. (3) Sau khi xây xong `mp`, kiểm tra `mp.get(x.random) is y.random` cho mọi cặp (với `random` `None` thì cả hai phải `None`).

## 19. Đáp án tham khảo

In [29]:
# Bài 1
data = [[3, 2], [4, None], [5, 0]]
t = build_random_tree(data)
print("random của gốc có val =", t.random.val)
ans1 = serialize(Solution().copyRandomBinaryTree(t))
print(ans1)
assert t.random.val == 5 and ans1 == data

# Bài 2: ghi lại thứ tự tạo
order_created = []
def clone_dfs_log(root):
    seen = {}
    def dfs(old):
        if old is None:
            return None
        if old in seen:
            return seen[old]
        new = NodeCopy(old.val); seen[old] = new; order_created.append(old.val)
        new.left, new.right, new.random = dfs(old.left), dfs(old.right), dfs(old.random)
        return new
    return dfs(root)
clone_dfs_log(build_random_tree(examples[2]))
print("thứ tự tạo:", order_created)
assert order_created == [1, 2, 4, 5, 3, 6, 7]

random của gốc có val = 5
[[3, 2], [4, None], [5, 0]]
thứ tự tạo: [1, 2, 4, 5, 3, 6, 7]


In [30]:
# Bài 3
def count_self_random_sol(root):
    return sum(1 for x in collect(root) if x.random is x)

assert count_self_random_sol(build_random_tree(examples[1])) == 1
assert count_self_random_sol(build_random_tree([[5, 0]])) == 1
assert count_self_random_sol(build_random_tree(examples[0])) == 0
assert count_self_random_sol(None) == 0

# Bài 4
def copy_random_list_sol(head):
    mp = {}
    cur = head
    while cur:
        mp[cur] = LNode(cur.val)
        cur = cur.next
    for o, c in mp.items():
        c.next = mp.get(o.next)
        c.random = mp.get(o.random)
    return mp.get(head)

def build_list(pairs):
    nodes = [LNode(v) for v, _ in pairs]
    for i, (v, r) in enumerate(pairs):
        nodes[i].next = nodes[i + 1] if i + 1 < len(nodes) else None
        nodes[i].random = nodes[r] if r is not None else None
    return nodes[0] if nodes else None

def list_to_pairs(head):
    nodes = []
    cur = head
    while cur:
        nodes.append(cur); cur = cur.next
    pos = {id(n): i for i, n in enumerate(nodes)}
    return [[n.val, pos.get(id(n.random)) if n.random else None] for n in nodes]

pairs = [[7, None], [13, 0], [11, 4], [10, 2], [1, 0]]
h = build_list(pairs)
h2 = copy_random_list_sol(h)
assert list_to_pairs(h2) == pairs
a, b = h, h2
while a:
    assert a is not b
    a, b = a.next, b.next
assert copy_random_list_sol(None) is None
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
# Bài 5
def is_deep_copy_pair_sol(a, b):
    if a is None or b is None:
        return a is None and b is None
    mp = {}
    q = deque([(a, b)])
    while q:
        x, y = q.popleft()
        if (x is None) != (y is None):
            return False
        if x is None:
            continue
        if x is y or x.val != y.val:
            return False
        mp[x] = y
        q.append((x.left, y.left))
        q.append((x.right, y.right))
    b_ids = {id(y) for y in mp.values()}
    if b_ids & {id(x) for x in mp}:
        return False
    for x, y in mp.items():
        if x.random is None:
            if y.random is not None:
                return False
        elif mp.get(x.random) is not y.random:
            return False
    return True

rng = random.Random(5)
for _ in range(200):
    orig = random_tree(rng, rng.randint(0, 25))
    good = Solution().copyRandomBinaryTree(orig)
    assert is_deep_copy_pair_sol(orig, good)
    if orig is not None:
        assert not is_deep_copy_pair_sol(orig, orig)
        assert not is_deep_copy_pair_sol(orig, clone_wrong1(orig)) or all(x.random is None for x in collect(orig))
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DFS memo (deep copy có con trỏ phụ):**

```python
seen = {}
def dfs(old):
    if old is None:
        return None
    if old in seen:
        return seen[old]
    new = NodeCopy(old.val)
    seen[old] = new              # ghi sổ TRƯỚC khi đệ quy
    new.left = dfs(old.left)
    new.right = dfs(old.right)
    new.random = dfs(old.random)
    return new
return dfs(root)
```

**Template hai lượt:**

```python
mp = {}                                   # lượt 1: tạo hết
for old in all_nodes(root): mp[old] = NodeCopy(old.val)
for old, new in mp.items():               # lượt 2: nối
    new.left, new.right, new.random = mp.get(old.left), mp.get(old.right), mp.get(old.random)
return mp.get(root)
```

**Checklist trước khi submit:**
- [ ] Tạo nút bằng **`NodeCopy`**, không phải `Node`.
- [ ] `random` của bản sao trỏ vào **bản sao** (tra `seen`), không trỏ về cây gốc.
- [ ] Khoá `dict` là **đối tượng nút**, không phải `val`.
- [ ] Ghi sổ **ngay sau khi tạo**, trước khi đệ quy (chặn chu trình, nút tự trỏ).
- [ ] Cây rỗng ⇒ trả `None`.
- [ ] Đúng chữ ký: `copyRandomBinaryTree(self, root)`.